In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
catalog = "vp_rp20040126"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# --- gold.dim_date: one row per day of April 2026 ---
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_date AS
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
    d AS calendar_date,
    date_format(d, 'EEEE') AS day_name,
    weekofyear(d) AS iso_week,
    CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS is_weekend
FROM (SELECT explode(sequence(to_date('2026-04-01'), to_date('2026-04-30'), interval 1 day)) AS d)
ORDER BY date_key
""")

# --- gold.dim_station: all columns of silver.stations ---
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_station AS
SELECT * FROM {catalog}.silver.stations
""")

# --- gold.dim_charger: chargers minus last_changed_snapshot_date, plus station city ---
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_charger AS
SELECT
    c.charger_id,
    c.station_id,
    c.connector_type,
    c.max_kw,
    c.is_active,
    c.commissioned_on,
    s.city AS station_city
FROM {catalog}.silver.chargers c
LEFT JOIN {catalog}.silver.stations s ON c.station_id = s.station_id
""")

# --- gold.dim_member: member_sk BIGINT (unique, NOT NULL) + all silver.members columns ---
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_member AS
SELECT
    CAST(row_number() OVER (ORDER BY member_id, effective_from) AS BIGINT) AS member_sk,
    member_id,
    full_name,
    email,
    phone,
    city,
    plan,
    joined_on,
    effective_from,
    effective_to,
    is_current
FROM {catalog}.silver.members
ORDER BY member_sk
""")

# Enforce NOT NULL on member_sk (row_number always produces non-null values)
spark.sql(f"ALTER TABLE {catalog}.gold.dim_member ALTER COLUMN member_sk SET NOT NULL")

In [0]:
# --- gold.fact_sessions: one row per silver.sessions row with billing ---
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.fact_sessions AS
WITH faults AS (
    -- Pre-aggregate fault counts per session
    SELECT session_id, COUNT(*) AS fault_count
    FROM {catalog}.silver.session_faults
    GROUP BY session_id
),
joined AS (
    -- Join sessions with dim_charger (station_id + connector_type),
    -- tariffs (rate_per_kwh, idle fees), dim_member (member_sk, plan), faults
    SELECT
        s.session_id,
        s.charger_id,
        s.member_id,
        s.session_date,
        s.status,
        s.time_band,
        s.energy_kwh,
        s.duration_minutes,
        s.idle_minutes,
        s.amount_paid,
        dc.station_id,
        m.member_sk,
        CASE WHEN s.member_id IS NULL THEN 'GUEST' ELSE m.plan END AS plan_at_session,
        t.rate_per_kwh,
        t.free_idle_minutes,
        t.idle_fee_per_min,
        COALESCE(f.fault_count, 0) AS fault_count
    FROM {catalog}.silver.sessions s
    LEFT JOIN {catalog}.gold.dim_charger dc ON s.charger_id = dc.charger_id
    LEFT JOIN {catalog}.silver.tariffs t
        ON dc.connector_type = t.connector_type AND s.time_band = t.time_band
    LEFT JOIN {catalog}.gold.dim_member m
        ON s.member_id = m.member_id
        AND s.session_date >= m.effective_from
        AND (s.session_date < m.effective_to OR m.effective_to IS NULL)
    LEFT JOIN faults f ON s.session_id = f.session_id
),
computed AS (
    -- Compute raw billing values (applied to all rows; zeroed later for non-COMPLETED)
    SELECT
        *,
        ROUND(energy_kwh * rate_per_kwh, 2) AS _energy_charge,
        ROUND(
            ROUND(energy_kwh * rate_per_kwh, 2) *
            CASE plan_at_session
                WHEN 'PRO'   THEN 0.10
                WHEN 'FLEET' THEN 0.15
                ELSE 0.00
            END,
            2
        ) AS _plan_discount,
        ROUND(
            GREATEST(idle_minutes - COALESCE(free_idle_minutes, 0), 0) *
            COALESCE(idle_fee_per_min, 0),
            2
        ) AS _idle_fee
    FROM joined
)
SELECT
    session_id,
    CAST(date_format(session_date, 'yyyyMMdd') AS INT) AS date_key,
    charger_id,
    station_id,
    member_id,
    status,
    time_band,
    CAST(energy_kwh   AS DECIMAL(12,3)) AS energy_kwh,
    duration_minutes,
    idle_minutes,
    CAST(amount_paid  AS DECIMAL(12,2)) AS amount_paid,
    member_sk,
    plan_at_session,
    CAST(rate_per_kwh AS DECIMAL(12,2)) AS rate_per_kwh,
    -- Billing: COMPLETED only; 0.00 for all other statuses
    CASE WHEN status = 'COMPLETED'
         THEN CAST(_energy_charge AS DECIMAL(12,2))
         ELSE CAST(0.00 AS DECIMAL(12,2)) END AS energy_charge,
    CASE WHEN status = 'COMPLETED'
         THEN CAST(_plan_discount AS DECIMAL(12,2))
         ELSE CAST(0.00 AS DECIMAL(12,2)) END AS plan_discount,
    CASE WHEN status = 'COMPLETED'
         THEN CAST(_idle_fee AS DECIMAL(12,2))
         ELSE CAST(0.00 AS DECIMAL(12,2)) END AS idle_fee,
    CASE WHEN status = 'COMPLETED'
         THEN CAST(_energy_charge - _plan_discount + _idle_fee AS DECIMAL(12,2))
         ELSE CAST(0.00 AS DECIMAL(12,2)) END AS billed_amount,
    CASE WHEN status = 'COMPLETED' THEN
        CASE WHEN abs(amount_paid - (_energy_charge - _plan_discount + _idle_fee)) <= 0.50
             THEN 'OK' ELSE 'MISMATCH' END
    ELSE NULL END AS billing_status,
    fault_count
FROM computed
""")

# display(spark.table(f"{catalog}.gold.fact_sessions"))

In [0]:
# # --- Rebuild silver.tariffs with uppercased connector_type (fixes OFF_PEAK join) ---
# spark.sql("""
# CREATE OR REPLACE TABLE vp_rp20040126.silver.tariffs AS
# SELECT
#     upper(trim(connector_type)) AS connector_type,
#     upper(trim(time_band)) AS time_band,
#     CAST(rate_per_kwh AS DECIMAL(10,2)) AS rate_per_kwh,
#     CAST(idle_fee_per_min AS DECIMAL(10,2)) AS idle_fee_per_min,
#     CAST(free_idle_minutes AS INT) AS free_idle_minutes
# FROM vp_rp20040126.bronze.tariffs
# """)
# spark.sql("SELECT connector_type, time_band, rate_per_kwh FROM vp_rp20040126.silver.tariffs ORDER BY connector_type, time_band").show()

# # --- Verify E2 results ---
# print("=== Tariff coverage ===")
# spark.sql("SELECT connector_type, time_band, rate_per_kwh, free_idle_minutes, idle_fee_per_min FROM vp_rp20040126.silver.tariffs ORDER BY connector_type, time_band").show(truncate=False)

# print("=== Billing status distribution ===")
# spark.sql("""
# SELECT status, billing_status, COUNT(*) AS cnt
# FROM vp_rp20040126.gold.fact_sessions
# GROUP BY status, billing_status
# ORDER BY status, billing_status
# """).show()

# print("=== Rate coverage by time_band ===")
# spark.sql("""
# SELECT time_band, 
#        SUM(CASE WHEN rate_per_kwh IS NOT NULL THEN 1 ELSE 0 END) AS has_rate,
#        SUM(CASE WHEN rate_per_kwh IS NULL THEN 1 ELSE 0 END) AS missing_rate
# FROM vp_rp20040126.gold.fact_sessions
# GROUP BY time_band
# """).show()

# print("=== Sample COMPLETED+OK sessions ===")
# spark.sql("""
# SELECT session_id, charger_id, time_band, plan_at_session, energy_kwh, rate_per_kwh,
#        energy_charge, plan_discount, idle_fee, billed_amount, amount_paid, billing_status
# FROM vp_rp20040126.gold.fact_sessions
# WHERE billing_status = 'OK'
# LIMIT 5
# """).show(truncate=False)

In [0]:
# --- E3: gold.agg_daily_station ---
# One row per station per date (including zero-session days), COMPLETED-only money/energy
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.agg_daily_station AS
WITH fact_with_date AS (
    SELECT *, to_date(CAST(date_key AS STRING), 'yyyyMMdd') AS session_date
    FROM {catalog}.gold.fact_sessions
),
date_range AS (
    SELECT explode(sequence(
        (SELECT min(session_date) FROM fact_with_date),
        (SELECT max(session_date) FROM fact_with_date),
        interval 1 day
    )) AS session_date
),
station_chargers AS (
    SELECT station_id, COUNT(*) AS num_chargers
    FROM {catalog}.gold.dim_charger
    GROUP BY station_id
),
daily_agg AS (
    SELECT
        session_date,
        station_id,
        SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_sessions,
        SUM(CASE WHEN status = 'FAILED'   THEN 1 ELSE 0 END) AS failed_sessions,
        SUM(CASE WHEN status = 'COMPLETED' THEN energy_kwh   ELSE 0 END) AS energy_kwh,
        SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END) AS billed_amount,
        SUM(CASE WHEN status = 'COMPLETED' THEN amount_paid  ELSE 0 END) AS amount_paid,
        SUM(CASE WHEN billing_status = 'MISMATCH' THEN 1 ELSE 0 END) AS mismatch_sessions,
        SUM(fault_count) AS fault_count,
        SUM(CASE WHEN status = 'COMPLETED' THEN duration_minutes ELSE 0 END) AS completed_duration
    FROM fact_with_date
    GROUP BY session_date, station_id
)
SELECT
    d.session_date,
    s.station_id,
    s.city,
    CAST(COALESCE(a.completed_sessions, 0) AS INT)       AS completed_sessions,
    CAST(COALESCE(a.failed_sessions, 0)    AS INT)       AS failed_sessions,
    CAST(COALESCE(a.energy_kwh, 0)     AS DECIMAL(12,3)) AS energy_kwh,
    CAST(COALESCE(a.billed_amount, 0)   AS DECIMAL(12,2)) AS billed_amount,
    CAST(COALESCE(a.amount_paid, 0)     AS DECIMAL(12,2)) AS amount_paid,
    CAST(COALESCE(a.mismatch_sessions, 0) AS INT)       AS mismatch_sessions,
    CAST(COALESCE(a.fault_count, 0)      AS INT)       AS fault_count,
    CAST(
        CASE WHEN COALESCE(sc.num_chargers, 0) = 0 THEN 0.00
             ELSE ROUND(100.0 * COALESCE(a.completed_duration, 0) / (sc.num_chargers * 1440), 2)
        END AS DECIMAL(5,2)
    ) AS utilisation_pct
FROM date_range d
CROSS JOIN {catalog}.gold.dim_station s
LEFT JOIN daily_agg a  ON d.session_date = a.session_date AND s.station_id = a.station_id
LEFT JOIN station_chargers sc ON s.station_id = sc.station_id
ORDER BY d.session_date, s.station_id
""")

# --- E4: gold.v_kpi_summary (one-row view) ---
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS
SELECT
    CAST(SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END) AS INT) AS completed_sessions,
    CAST(SUM(CASE WHEN status = 'FAILED'   THEN 1 ELSE 0 END) AS INT) AS failed_sessions,
    CAST(CASE
        WHEN SUM(CASE WHEN status IN ('COMPLETED','FAILED') THEN 1 ELSE 0 END) = 0 THEN 0.00
        ELSE ROUND(
            SUM(CASE WHEN status = 'COMPLETED' THEN 1 ELSE 0 END) * 100.0 /
            SUM(CASE WHEN status IN ('COMPLETED','FAILED') THEN 1 ELSE 0 END),
            2
        )
    END AS DECIMAL(5,2)) AS success_rate_pct,
    CAST(SUM(CASE WHEN status = 'COMPLETED' THEN energy_kwh   ELSE 0 END) AS DECIMAL(12,3)) AS total_energy_kwh,
    CAST(SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END) AS DECIMAL(12,2)) AS total_billed,
    CAST(SUM(CASE WHEN status = 'COMPLETED' THEN amount_paid  ELSE 0 END) AS DECIMAL(12,2)) AS total_paid,
    CAST(
        SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END) -
        SUM(CASE WHEN status = 'COMPLETED' THEN amount_paid  ELSE 0 END)
        AS DECIMAL(12,2)
    ) AS revenue_leakage,
    CAST(SUM(CASE WHEN billing_status = 'MISMATCH' THEN 1 ELSE 0 END) AS INT) AS mismatch_sessions,
    CAST(COUNT(DISTINCT CASE WHEN status = 'COMPLETED' THEN member_id END) AS INT) AS active_members
FROM {catalog}.gold.fact_sessions
""")

# display(spark.table(f"{catalog}.gold.agg_daily_station"))
# display(spark.table(f"{catalog}.gold.v_kpi_summary"))

In [0]:
# --- E5: ops.reconciliation_results ---
# Each run appends 3 rows: SESSION_COUNT, ENERGY_KWH, CURRENT_MEMBERS
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
    run_ts        TIMESTAMP,
    check_name    STRING,
    source_value  DECIMAL(18,3),
    target_value  DECIMAL(18,3),
    status        STRING
) USING DELTA
""")

spark.sql(f"""
INSERT INTO {catalog}.ops.reconciliation_results
SELECT
    current_timestamp()                       AS run_ts,
    t.check_name,
    t.source_value,
    t.target_value,
    CASE WHEN abs(t.source_value - t.target_value) < 0.01
         THEN 'PASS' ELSE 'FAIL' END           AS status
FROM (
    -- 1) SESSION_COUNT: distinct standardised session_id in Bronze
    --    vs silver.sessions + silver.quarantine_sessions rows
    SELECT
        'SESSION_COUNT' AS check_name,
        CAST((SELECT COUNT(DISTINCT nullif(trim(upper(session_id)), ''))
             FROM {catalog}.bronze.sessions) AS DECIMAL(18,3)) AS source_value,
        CAST((SELECT COUNT(*) FROM {catalog}.silver.sessions)
           + (SELECT COUNT(*) FROM {catalog}.silver.quarantine_sessions)
             AS DECIMAL(18,3)) AS target_value

    UNION ALL

    -- 2) ENERGY_KWH: COMPLETED energy in silver.sessions
    --    vs total energy_kwh in gold.agg_daily_station
    SELECT
        'ENERGY_KWH' AS check_name,
        CAST((SELECT COALESCE(SUM(energy_kwh), 0) FROM {catalog}.silver.sessions
             WHERE status = 'COMPLETED') AS DECIMAL(18,3)) AS source_value,
        CAST((SELECT COALESCE(SUM(energy_kwh), 0) FROM {catalog}.gold.agg_daily_station)
             AS DECIMAL(18,3)) AS target_value

    UNION ALL

    -- 3) CURRENT_MEMBERS: distinct member_id in silver.members
    --    vs current rows in gold.dim_member
    SELECT
        'CURRENT_MEMBERS' AS check_name,
        CAST((SELECT COUNT(DISTINCT member_id) FROM {catalog}.silver.members)
             AS DECIMAL(18,3)) AS source_value,
        CAST((SELECT COUNT(*) FROM {catalog}.gold.dim_member WHERE is_current = true)
             AS DECIMAL(18,3)) AS target_value
) t
""")

# display(spark.sql(f"SELECT * FROM {catalog}.ops.reconciliation_results ORDER BY run_ts DESC, check_name"))